# 7. EVALUATION
Questo notebook aggrega le metriche prestazionali e produce grafici utili alla comprensione e alla rappresentazione dei risultati dell'intera pipeline modulare:
**YOLO Detection → SAM Segmentation → Player Exclusion → Green Screen**

---
## Sezione 1 · Setup & Import

In [ ]:
import os, cv2, torch, json, subprocess, yaml
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

subprocess.run(['pip', 'install', '-q', 'ultralytics==8.3.86', 'matplotlib', 'numpy<2', 'scipy', 'tqdm'], check=True)
from ultralytics import YOLO, SAM
from tqdm import tqdm

plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor': '#FAFAFA',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 12
})

# ── Percorsi Progetto ─────────────────────────────────────────────────────────
BASE = Path('/teamspace/studios/this_studio/project-sbs')
DATASET_YOLO  = BASE / 'datasets' / 'dataset_yolo'
DATASET_SAM   = BASE / 'datasets' / 'dataset_sam'
RUNS_DIR      = BASE / 'runs' / 'yolo_banner_training'
YOLO_BEST     = RUNS_DIR / 'weights' / 'best.pt'
SAM_MODEL_PATH   = BASE / 'model_weights' / 'sam' / 'sam2.1_l.pt'
PERSON_MODEL_PATH = BASE / 'model_weights' / 'yolo_seg' / 'yolo11n-seg.pt'
YAML_PATH     = DATASET_YOLO / 'data.yaml'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'📂 Base: {BASE}')
print(f'🖥️ Device: {DEVICE}')
print(f'🏋️ YOLO best.pt: {"✅" if YOLO_BEST.exists() else "❌"}')
print(f'🏋️ SAM model:    {"✅" if SAM_MODEL_PATH.exists() else "❌"}')
print(f'🏋️ YOLO-Seg:     {"✅" if PERSON_MODEL_PATH.exists() else "❌"}')

---
## Sezione 2 · Analisi del Dataset

### 2a · Distribuzione Immagini: Train / Val / Test

In [ ]:
splits = ['train', 'val', 'test']
colors_split = ['#1976D2', '#F57C00', '#388E3C']
counts = {}
for s in splits:
    img_dir = DATASET_YOLO / 'images' / s
    counts[s] = len(list(img_dir.glob('*.*'))) if img_dir.exists() else 0

total = sum(counts.values())
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(counts.keys(), counts.values(), color=colors_split, edgecolor='black', linewidth=0.8)
for bar, (s, c) in zip(bars, counts.items()):
    pct = c / total * 100 if total > 0 else 0
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total*0.01,
            f'{c}  ({pct:.1f}%)', ha='center', fontweight='bold', fontsize=13)
ax.set_title(f'Distribuzione Dataset YOLO — Totale: {total} immagini', fontsize=16, fontweight='bold')
ax.set_ylabel('Numero Immagini', fontsize=13)
ax.set_xlabel('Split', fontsize=13)
plt.tight_layout()
plt.show()

### 2b · Distribuzione Dimensioni Immagini

In [ ]:
widths, heights = [], []
for s in splits:
    img_dir = DATASET_YOLO / 'images' / s
    if not img_dir.exists(): continue
    for img_p in img_dir.glob('*.*'):
        img = cv2.imread(str(img_p))
        if img is not None:
            h, w = img.shape[:2]
            widths.append(w)
            heights.append(h)

if widths:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].hist(widths, bins=30, color='#1976D2', edgecolor='black', alpha=0.85)
    axes[0].axvline(np.mean(widths), color='red', linestyle='--', lw=2, label=f'Media = {np.mean(widths):.0f} px')
    axes[0].set_title('Distribuzione Larghezza Immagini', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Larghezza (px)'); axes[0].set_ylabel('Frequenza'); axes[0].legend()

    axes[1].hist(heights, bins=30, color='#F57C00', edgecolor='black', alpha=0.85)
    axes[1].axvline(np.mean(heights), color='red', linestyle='--', lw=2, label=f'Media = {np.mean(heights):.0f} px')
    axes[1].set_title('Distribuzione Altezza Immagini', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Altezza (px)'); axes[1].set_ylabel('Frequenza'); axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print('❌ Nessuna immagine trovata nel dataset YOLO.')

### 2c · Distribuzione Dimensioni Bounding Box (Annotazioni YOLO)

In [ ]:
bw_list, bh_list = [], []
for s in splits:
    lbl_dir = DATASET_YOLO / 'labels' / s
    if not lbl_dir.exists(): continue
    for lbl_p in lbl_dir.glob('*.txt'):
        with open(lbl_p) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    bw_list.append(float(parts[3]))
                    bh_list.append(float(parts[4]))

if bw_list:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].hist(bw_list, bins=30, color='#388E3C', edgecolor='black', alpha=0.85)
    axes[0].axvline(np.mean(bw_list), color='red', linestyle='--', lw=2, label=f'Media = {np.mean(bw_list):.3f}')
    axes[0].set_title('Larghezza BBox (relativa)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Larghezza Relativa (0–1)'); axes[0].set_ylabel('Frequenza'); axes[0].legend()

    axes[1].hist(bh_list, bins=30, color='#D32F2F', edgecolor='black', alpha=0.85)
    axes[1].axvline(np.mean(bh_list), color='blue', linestyle='--', lw=2, label=f'Media = {np.mean(bh_list):.3f}')
    axes[1].set_title('Altezza BBox (relativa)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Altezza Relativa (0–1)'); axes[1].set_ylabel('Frequenza'); axes[1].legend()
    plt.suptitle(f'Annotazioni YOLO — Totale: {len(bw_list)} bounding box', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('❌ Nessuna annotazione trovata.')

---
## Sezione 3 · Metriche YOLO (Validazione dai Pesi)
Anche senza i log di training, possiamo ricalcolare tutte le metriche di detection eseguendo `model.val()` sui pesi addestrati (`best.pt`).

### 3a · Validazione YOLO (Precision, Recall, mAP)

In [ ]:
# ── Assicuriamoci che il data.yaml esista ──────────────────────────────────────
if not YAML_PATH.exists():
    yaml_content = {
        'path': str(DATASET_YOLO),
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'nc': 1,
        'names': ['banner']
    }
    YAML_PATH.write_text(yaml.dump(yaml_content, default_flow_style=False), encoding='utf-8')
    print(f'✅ YAML creato: {YAML_PATH}')

# ── Validazione ────────────────────────────────────────────────────────────────
yolo_model = YOLO(str(YOLO_BEST))
val_results = yolo_model.val(data=str(YAML_PATH), imgsz=640, batch=16, split='test', verbose=False)

# Estrai metriche
precision = val_results.box.mp     # mean precision
recall    = val_results.box.mr     # mean recall
map50     = val_results.box.map50  # mAP@0.5
map50_95  = val_results.box.map    # mAP@0.5:0.95

print(f'\n📊 Risultati Validazione YOLO (Test Set)')
print(f'   Precision:     {precision:.4f}')
print(f'   Recall:        {recall:.4f}')
print(f'   mAP@0.5:       {map50:.4f}')
print(f'   mAP@0.5:0.95:  {map50_95:.4f}')

### 3b · Grafico Metriche YOLO

In [ ]:
metric_names = ['Precision', 'Recall', 'mAP@0.5', 'mAP@0.5:0.95']
metric_vals  = [precision, recall, map50, map50_95]
metric_colors = ['#1976D2', '#388E3C', '#F57C00', '#D32F2F']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(metric_names, metric_vals, color=metric_colors, edgecolor='black', linewidth=0.8, width=0.55)
for bar, val in zip(bars, metric_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
            f'{val:.4f}', ha='center', fontweight='bold', fontsize=13)
ax.set_ylim(0, 1.1)
ax.set_title('Metriche di Detection — YOLOv12n (Test Set)', fontsize=16, fontweight='bold')
ax.set_ylabel('Valore', fontsize=13)
ax.axhline(y=1.0, color='gray', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

### 3c · Curva Precision-Recall e Distribuzione Confidenza

In [ ]:
# ── Calcola P-R curve e confidenza su tutto il test set ────────────────────────
test_dir = DATASET_YOLO / 'images' / 'test'
test_imgs = sorted(test_dir.glob('*.*')) if test_dir.exists() else []

all_confs = []
per_img_detections = []

for ip in tqdm(test_imgs, desc='Inference YOLO'):
    res = yolo_model(str(ip), verbose=False, conf=0.1, imgsz=640)
    n_det = len(res[0].boxes)
    per_img_detections.append(n_det)
    if n_det > 0:
        all_confs.extend(res[0].boxes.conf.cpu().numpy().tolist())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Confidenza
if all_confs:
    axes[0].hist(all_confs, bins=30, color='#1976D2', edgecolor='black', alpha=0.85)
    axes[0].axvline(np.mean(all_confs), color='red', linestyle='--', lw=2,
                    label=f'Media = {np.mean(all_confs):.3f}')
    axes[0].set_title('Distribuzione Confidenza Detection', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Confidenza'); axes[0].set_ylabel('Frequenza'); axes[0].legend()

# Detection per immagine
axes[1].bar(range(len(per_img_detections)), per_img_detections, color='#F57C00', edgecolor='black', linewidth=0.3)
axes[1].axhline(np.mean(per_img_detections), color='red', linestyle='--', lw=2,
                label=f'Media = {np.mean(per_img_detections):.1f}')
axes[1].set_title('Banner Rilevati per Immagine (Test Set)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Indice Immagine'); axes[1].set_ylabel('N. Detection'); axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\n📊 Confidenza media: {np.mean(all_confs):.4f}' if all_confs else '')
print(f'📊 Detection medie/immagine: {np.mean(per_img_detections):.2f}')
print(f'📊 Immagini senza detection: {per_img_detections.count(0)}/{len(per_img_detections)}')

---
## Sezione 4 · Metriche SAM — Intersection over Union (IoU)

### 4a · Calcolo mIoU, Precision e Recall

In [ ]:
VAL_SAM_DIR = DATASET_SAM / 'val'
ious = []
precisions = []
recalls = []
iou_records = []  # (iou, img_path, json_path)

if not VAL_SAM_DIR.exists():
    print('❌ Cartella validation SAM non trovata.')
else:
    sam_model_eval = SAM(str(SAM_MODEL_PATH))
    json_files = sorted(VAL_SAM_DIR.glob('*.json'))
    print(f'📊 Inizio valutazione SAM su {len(json_files)} file JSON...\n')

    for jpath in tqdm(json_files, desc='Calcolo Metriche'):
        idx = jpath.stem
        img_p = VAL_SAM_DIR / f'frame{idx}.jpg'
        if not img_p.exists(): img_p = VAL_SAM_DIR / f'frame{idx}.png'
        if not img_p.exists(): continue

        data = json.loads(jpath.read_text())
        regions = data.get('regions', [])
        if not regions: continue

        img = cv2.imread(str(img_p))
        if img is None: continue

        gt_mask = np.zeros(img.shape[:2], dtype=np.uint8)
        all_x = regions[0]['all_points_x']
        all_y = regions[0]['all_points_y']
        pts = np.column_stack((all_x, all_y)).astype(np.int32)
        cv2.fillPoly(gt_mask, [pts], 1)
        x, y, w, h = cv2.boundingRect(pts)
        gt_box = [x, y, x + w, y + h]

        sam_res = sam_model_eval(img, bboxes=[gt_box], verbose=False)
        if not sam_res[0].masks: continue

        m_data = sam_res[0].masks.data.cpu().numpy()
        pred_mask = np.zeros(img.shape[:2], dtype=np.uint8)
        for m in m_data:
            if m.shape != img.shape[:2]:
                m = cv2.resize(m, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
            pred_mask = np.maximum(pred_mask, (m > 0.5).astype(np.uint8))

        inter = np.logical_and(gt_mask, pred_mask).sum()
        union = np.logical_or(gt_mask, pred_mask).sum()
        sum_pred = pred_mask.sum()
        sum_gt   = gt_mask.sum()

        if union > 0:
            iou_val = inter / union
            ious.append(iou_val)
            iou_records.append((iou_val, str(img_p), str(jpath)))
            
            # Precision & Recall
            precisions.append(inter / sum_pred if sum_pred > 0 else 0)
            recalls.append(inter / sum_gt if sum_gt > 0 else 0)

    if ious:
        mean_iou = np.mean(ious)
        mean_prec = np.mean(precisions)
        mean_rec = np.mean(recalls)
        
        print(f'\n✅ Valutazione completata su {len(ious)} immagini.')
        print(f'⭐ Mean IoU:       {mean_iou:.4f} ({mean_iou*100:.2f}%)')
        print(f'⭐ Mean Precision: {mean_prec:.4f} ({mean_prec*100:.2f}%)')
        print(f'⭐ Mean Recall:    {mean_rec:.4f} ({mean_rec*100:.2f}%)')

        fig, ax = plt.subplots(figsize=(12, 5))
        ax.hist(ious, bins=25, color='#1976D2', edgecolor='black', alpha=0.85)
        ax.axvline(mean_iou, color='red', linestyle='--', lw=2.5, label=f'Media IoU = {mean_iou:.4f}')
        ax.set_title('Distribuzione IoU per Immagine — SAM 2.1 Large', fontsize=16, fontweight='bold')
        ax.set_xlabel('IoU', fontsize=13); ax.set_ylabel('Frequenza', fontsize=13); ax.legend(fontsize=12)
        plt.tight_layout()
        plt.show()
    else:
        print('⚠️ Nessun dato metrico computato.')

### 4b · Box Plot IoU con Statistiche

In [ ]:
if ious:
    fig, ax = plt.subplots(figsize=(8, 6))
    bp = ax.boxplot(ious, vert=True, patch_artist=True, widths=0.5,
                    boxprops=dict(facecolor='#BBDEFB', edgecolor='#1565C0', linewidth=1.5),
                    medianprops=dict(color='#D32F2F', linewidth=2.5),
                    whiskerprops=dict(color='#1565C0', linewidth=1.5),
                    capprops=dict(color='#1565C0', linewidth=1.5),
                    flierprops=dict(marker='o', markerfacecolor='#F57C00', markersize=6, alpha=0.7))
    ax.set_title('Box Plot IoU — SAM 2.1 Large', fontsize=16, fontweight='bold')
    ax.set_ylabel('IoU', fontsize=13)
    ax.set_xticklabels(['SAM 2.1 Large'], fontsize=13)

    stats_text = (f'Media: {np.mean(ious):.4f}\n'
                  f'Mediana: {np.median(ious):.4f}\n'
                  f'Min: {np.min(ious):.4f}\n'
                  f'Max: {np.max(ious):.4f}\n'
                  f'Std: {np.std(ious):.4f}')
    ax.text(1.35, np.median(ious), stats_text, fontsize=11,
            verticalalignment='center', bbox=dict(boxstyle='round,pad=0.5', facecolor='#E3F2FD', alpha=0.9))
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ Eseguire prima la cella 4a.')

### 4c · Confronto Metriche SAM (IoU vs Precision vs Recall)

In [ ]:
if ious:
    sam_metrics = ['Mean IoU', 'Mean Precision', 'Mean Recall']
    sam_vals  = [np.mean(ious), np.mean(precisions), np.mean(recalls)]
    sam_colors = ['#1976D2', '#7B1FA2', '#388E3C']

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(sam_metrics, sam_vals, color=sam_colors, edgecolor='black', linewidth=0.8, width=0.5)
    for bar, val in zip(bars, sam_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.4f}', ha='center', fontweight='bold', fontsize=13)
    ax.set_ylim(0, 1.1)
    ax.set_title('Riepilogo Metriche Segmentazione — SAM 2.1 Large', fontsize=16, fontweight='bold')
    ax.set_ylabel('Valore', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ Eseguire prima la cella 4a.')

### 4d · Confronto Visivo: Migliori vs Peggiori IoU

In [ ]:
if iou_records:
    sorted_records = sorted(iou_records, key=lambda x: x[0])
    worst_n = sorted_records[:3]
    best_n  = sorted_records[-3:]

    fig, axes = plt.subplots(2, 3, figsize=(26, 14))

    for i, (iou_val, img_path, jpath) in enumerate(reversed(best_n)):
        img = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        data = json.loads(Path(jpath).read_text())
        regions = data.get('regions', [])
        gt_mask = np.zeros(img.shape[:2], dtype=np.uint8)
        pts = np.column_stack((regions[0]['all_points_x'], regions[0]['all_points_y'])).astype(np.int32)
        cv2.fillPoly(gt_mask, [pts], 1)
        overlay = img_rgb.copy()
        overlay[gt_mask == 1] = [0, 200, 0]
        axes[0][i].imshow(cv2.addWeighted(img_rgb, 0.5, overlay, 0.5, 0))
        axes[0][i].set_title(f'✅ Best #{i+1} — IoU: {iou_val:.4f}', fontsize=13, fontweight='bold', color='#2E7D32')
        axes[0][i].axis('off')

    for i, (iou_val, img_path, jpath) in enumerate(worst_n):
        img = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        data = json.loads(Path(jpath).read_text())
        regions = data.get('regions', [])
        gt_mask = np.zeros(img.shape[:2], dtype=np.uint8)
        pts = np.column_stack((regions[0]['all_points_x'], regions[0]['all_points_y'])).astype(np.int32)
        cv2.fillPoly(gt_mask, [pts], 1)
        overlay = img_rgb.copy()
        overlay[gt_mask == 1] = [200, 0, 0]
        axes[1][i].imshow(cv2.addWeighted(img_rgb, 0.5, overlay, 0.5, 0))
        axes[1][i].set_title(f'❌ Worst #{i+1} — IoU: {iou_val:.4f}', fontsize=13, fontweight='bold', color='#C62828')
        axes[1][i].axis('off')

    plt.suptitle('Confronto Qualità Segmentazione SAM: Migliori vs Peggiori', fontsize=18, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()
else:
    print('⚠️ Eseguire prima la cella 4a.')

---
## Sezione 5 · Risultati Pipeline — Confronto Visivo

In [ ]:
from scipy.ndimage import median_filter

# Caricamento modelli (se non già caricati)
if 'sam_model_eval' not in dir():
    sam_model_eval = SAM(str(SAM_MODEL_PATH))
if 'person_model' not in dir():
    person_model = YOLO(str(PERSON_MODEL_PATH))

def get_players_mask(img_rgb, model):
    res = model(img_rgb, classes=[0], verbose=False, conf=0.2, imgsz=1024, retina_masks=True)
    mask = np.zeros(img_rgb.shape[:2], dtype=np.uint8)
    if len(res) > 0 and res[0].masks is not None:
        for m in res[0].masks.data.cpu().numpy():
            if m.shape != img_rgb.shape[:2]:
                m = cv2.resize(m, (img_rgb.shape[1], img_rgb.shape[0]), interpolation=cv2.INTER_LINEAR)
            mask = np.maximum(mask, (m > 0.5).astype(np.uint8) * 255)
    return mask

def run_pipeline(img_path, yolo_m, sam_m, seg_m):
    """Esegue la pipeline completa e restituisce (orig, yolo_viz, sam_viz, players_viz, gs_viz)."""
    yolo_res = yolo_m(str(img_path), verbose=False, conf=0.2, imgsz=640)
    raw_boxes = yolo_res[0].boxes.xyxy.cpu().numpy()
    if len(raw_boxes) == 0: return None

    sorted_boxes = sorted(raw_boxes.tolist(), key=lambda b: b[0])
    merged = []
    for b in sorted_boxes:
        if not merged: merged.append(list(b)); continue
        last = merged[-1]
        if b[0] <= last[2] + 600 and max(b[1], last[1]) < min(b[3], last[3]):
            last[0], last[1] = min(last[0], b[0]), min(last[1], b[1])
            last[2], last[3] = max(last[2], b[2]), max(last[3], b[3])
        else: merged.append(list(b))
    boxes = np.array(merged)

    img = cv2.imread(str(img_path))
    if img is None: return None
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # YOLO plot
    yolo_plot = img.copy()
    for b in boxes:
        x1, y1, x2, y2 = int(b[0]), int(b[1]), int(b[2]), int(b[3])
        cv2.rectangle(yolo_plot, (x1, y1), (x2, y2), (0, 255, 0), 4)
        label = 'Banner Box'
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 2.0, 4)
        cv2.rectangle(yolo_plot, (x1, max(0, y1 - th - 25)), (x1 + tw + 10, y1), (0, 255, 0), -1)
        cv2.putText(yolo_plot, label, (x1 + 5, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0,0,0), 4, cv2.LINE_AA)

    # SAM mask
    sam_res = sam_m(img_rgb, bboxes=boxes, verbose=False)
    sam_plot = sam_res[0].plot(boxes=False)
    for b in boxes:
        x1, y1, x2, y2 = int(b[0]), int(b[1]), int(b[2]), int(b[3])
        cv2.rectangle(sam_plot, (x1, y1), (x2, y2), (0, 0, 255), 4)
        lbl = 'Banner Pubblicitario'
        (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 2.0, 4)
        cv2.rectangle(sam_plot, (x1, max(0, y1 - th - 25)), (x1 + tw + 10, y1), (0, 0, 255), -1)
        cv2.putText(sam_plot, lbl, (x1 + 5, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 2.0, (255,255,255), 4, cv2.LINE_AA)

    masks = sam_res[0].masks.data.cpu().numpy()
    banner_mask = np.zeros(img.shape[:2], dtype=np.uint8)
    for m in masks:
        if m.shape != img.shape[:2]:
            m = cv2.resize(m, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_LINEAR)
        m_bin = (m > 0.5).astype(np.uint8) * 255
        contours, _ = cv2.findContours(m_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            if cv2.contourArea(cnt) > 500:
                cv2.drawContours(banner_mask, [cnt], -1, 255, -1)

    clip_mask = np.zeros_like(banner_mask)
    margin = 15
    for b in boxes:
        bx1, by1 = max(0, int(b[0])-margin), max(0, int(b[1])-margin)
        bx2, by2 = min(banner_mask.shape[1], int(b[2])+margin), min(banner_mask.shape[0], int(b[3])+margin)
        clip_mask[by1:by2, bx1:bx2] = 255
    banner_mask = cv2.bitwise_and(banner_mask, clip_mask)

    h_m, w_m = banner_mask.shape
    top_p = np.full(w_m, -1, dtype=np.int32)
    bot_p = np.full(w_m, -1, dtype=np.int32)
    for col in range(w_m):
        px = np.where(banner_mask[:, col] > 127)[0]
        if len(px) > 0: top_p[col], bot_p[col] = px[0], px[-1]
    valid = top_p >= 0
    if np.sum(valid) > 50:
        ts, bs = top_p.copy(), bot_p.copy()
        ts[valid] = median_filter(top_p[valid], size=301, mode='nearest')
        bs[valid] = median_filter(bot_p[valid], size=301, mode='nearest')
        sm = np.zeros_like(banner_mask)
        for col in range(w_m):
            if valid[col]: sm[ts[col]:bs[col]+1, col] = 255
        banner_mask = sm

    kc = cv2.getStructuringElement(cv2.MORPH_RECT, (41, 41))
    ko = cv2.getStructuringElement(cv2.MORPH_RECT, (11, 11))
    banner_mask = cv2.morphologyEx(banner_mask, cv2.MORPH_CLOSE, kc)
    banner_mask = cv2.morphologyEx(banner_mask, cv2.MORPH_OPEN, ko)
    banner_mask = cv2.GaussianBlur(banner_mask, (31, 31), 0)
    _, banner_mask = cv2.threshold(banner_mask, 127, 255, cv2.THRESH_BINARY)
    banner_mask = cv2.GaussianBlur(banner_mask, (3, 3), 0)

    # Player exclusion
    players = get_players_mask(img_rgb, seg_m)
    players_viz = players.copy()
    players = cv2.GaussianBlur(players, (11, 11), 0)

    ba = banner_mask.astype(float) / 255.0
    pa = players.astype(float) / 255.0
    fa = np.clip(ba - pa, 0, 1)[:, :, np.newaxis]
    gs = np.zeros_like(img); gs[:] = [0, 255, 0]
    final = (gs * fa + img * (1 - fa)).astype(np.uint8)

    return (
        cv2.cvtColor(img, cv2.COLOR_BGR2RGB),
        cv2.cvtColor(yolo_plot, cv2.COLOR_BGR2RGB),
        sam_plot,
        players_viz,
        cv2.cvtColor(final, cv2.COLOR_BGR2RGB)
    )

# ── Esecuzione su campione ─────────────────────────────────────────────────────
test_dir = DATASET_YOLO / 'images' / 'test'
if test_dir.exists():
    test_imgs = sorted(test_dir.glob('*.*'))
    import random
    samples = random.sample(test_imgs, min(5, len(test_imgs)))

    for img_p in samples:
        result = run_pipeline(img_p, yolo_model, sam_model_eval, person_model)
        if result is None:
            print(f'⚠️ {img_p.name}: Nessun banner rilevato, skip...')
            continue
        orig, yolo_viz, sam_viz, players_viz, gs_viz = result

        fig, axes = plt.subplots(2, 3, figsize=(28, 16))
        axes_flat = axes.flatten()
        
        titles = [
            '1. Originale', 
            '2. YOLO Detection (Banner Box)', 
            '3. SAM Segmentation (Banner Pub)', 
            '4. YOLO-Seg (Esclusione Giocatori)', 
            '5. Risultato Green Screen'
        ]
        images = [orig, yolo_viz, sam_viz, players_viz, gs_viz]
        cmaps = [None, None, None, 'gray', None]

        for i in range(len(titles)):
            axes_flat[i].imshow(images[i], cmap=cmaps[i])
            axes_flat[i].set_title(titles[i], fontsize=15, fontweight='bold')
            axes_flat[i].axis('off')
        
        axes_flat[-1].axis('off') # Nascondi l'ultimo box vuoto
        
        plt.suptitle(f'Pipeline Completa: {img_p.name}', fontsize=20, fontweight='bold')
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()
    print('\n✅ Confronto visivo completato.')
else:
    print('❌ Immagini di test non disponibili.')